In [1]:
import os, glob, shutil, zipfile, gdown
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorboard as tb
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks, optimizers, losses
%matplotlib inline

# Load the TensorBoard notebook extension
%load_ext tensorboard

# metrics
from sklearn import metrics

In [2]:
# Global Variables
FILE_ID = "1FOpBqxqZRkAkv1rlTPw1ZLME1wG_tJOv"
tf.keras.utils.set_random_seed(111)
height, width = 128, 128
channels = 3

In [3]:
# Utility Function

# function to download the dataset from the url
def download_dataset_from_url(url, file_path):
    try:
        gdown.download(url, file_path, quiet=False)
        print(f"Successfully downloaded to {file_path}")
    except Exception as e:
        print(f"Error downloading file: {e}")

# function to unzip the file
def unzip_file_shutil(zip_path, extract_path):
    try:
        os.makedirs(extract_path, exist_ok=True)
        shutil.unpack_archive(zip_path, extract_path)
        print(f"Successfully extracted to {extract_path}")
    except Exception as e:
        print(f"Error extracting file: {e}")

def deleteLogs(log_path):
    try:
        shutil.rmtree(log_path)
        print(f"Deleted logs from {log_path}")
    except Exception as e:
        print(f"Error deleting logs: {e}")

def generate_logs_Folder(logfolderName="", clear_folder=False):
    currentPath = os.getcwd()

    # clear existing eventLogs if clear_folder is True
    if clear_folder:
        deleteLogs(os.path.join(currentPath, "logs"))

    if not os.path.exists(os.path.join(currentPath, "logs")):
        os.mkdir("logs")

    if logfolderName:
        logfolderName = logfolderName + "_" + time.strftime("%Y%m%d-%H%M%S")
        logName = os.path.join(currentPath, "logs", logfolderName)
        if not os.path.exists(logName):
            os.mkdir(logName)
    else:
        logName = os.path.join(currentPath, "logs")
    
    return logName

def generate_logs_Name(fileName, logFormat=".log", clear_folder=False):
    folderName = generate_logs_Folder(fileName, clear_folder=clear_folder)
    fileName = fileName + "_" + time.strftime("%Y%m%d-%H%M%S") + logFormat
    return os.path.join(folderName, fileName)

In [4]:
def download_and_prep_dataset(FILE_ID):
    url = f'https://drive.google.com/uc?id={FILE_ID}'
    currentPath = os.getcwd()
    print("CWD: ", currentPath)
    dataSet_path = os.path.abspath(os.path.join(currentPath, "../dataSet/cloth_dataSet_small.zip"))
    extractPath = os.path.abspath(os.path.join(currentPath, "../unzipLoc/"))

    # create directory if not exists
    if not os.path.exists(os.path.abspath(os.path.join(currentPath, "../dataSet/"))):
        os.mkdir(os.path.abspath(os.path.join(currentPath, "../dataSet/")))

    if not os.path.exists(extractPath):
        os.mkdir(extractPath)

    # download the dataset
    if not os.path.exists(dataSet_path):
        download_dataset_from_url(url=url, file_path=dataSet_path)

    # unzip the dataset
    if os.listdir(extractPath) == []:
        unzip_file_shutil(zip_path=dataSet_path, extract_path=extractPath)
    
    # Load the dataset
    data_path = os.path.join(extractPath, "clothing-dataset-small")
    return data_path

In [6]:
# prepare the Dataset
dl_ext_data_path = download_and_prep_dataset(FILE_ID=FILE_ID)
print(dl_ext_data_path)

# Check the dataset Paths
train_path = os.path.join(dl_ext_data_path, "train")
test_path = os.path.join(dl_ext_data_path, "test")
val_path = os.path.join(dl_ext_data_path, "validation")

assert os.path.exists(train_path), "Train path does not exist"
assert os.path.exists(test_path), "Test path does not exist"
assert os.path.exists(val_path), "Val path does not exist"

# Load the dataset
Train_DS = tf.keras.utils.image_dataset_from_directory(directory=train_path, shuffle=True, image_size=(height, width))
Test_DS = tf.keras.utils.image_dataset_from_directory(directory=test_path, shuffle=True, image_size=(height, width))
Val_DS = tf.keras.utils.image_dataset_from_directory(directory=val_path, shuffle=True, image_size=(height, width))

# Scale the DataSet
Train_scaled = Train_DS.map(lambda x, y: (x/255.0, y))
Test_scaled = Test_DS.map(lambda x, y: (x/255.0, y))
Val_scaled = Val_DS.map(lambda x, y: (x/255.0, y))
classNames = Train_DS.class_names

CWD:  c:\Study\gitRepo\CV_Amazon_Cloth_Classify\notebooks
c:\Study\gitRepo\CV_Amazon_Cloth_Classify\unzipLoc\clothing-dataset-small
Found 3068 files belonging to 10 classes.
Found 372 files belonging to 10 classes.
Found 341 files belonging to 10 classes.


In [21]:
## Create CNN Model
# Base Line Model
def create_base_model(model_name="cnn_Model"):
    inpLayer = layers.Input(shape=(height, width, channels))
    conv2D_1 = layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same", activation='relu') (inpLayer)
    maxPool_1 = layers.MaxPooling2D() (conv2D_1)
    flatten = layers.Flatten() (maxPool_1)
    dense_1 = layers.Dense(units=128, activation='relu') (flatten)
    output = layers.Dense(units=len(classNames), activation='softmax') (dense_1)

    model = keras.Model(inputs=inpLayer, outputs=output, name=model_name)
    return model

def start_model_training(model, train_data, val_data, epochs=10):
    model.compile(optimizer=optimizers.Adam(),
                  loss=losses.SparseCategoricalCrossentropy(),
                  metrics=[keras.metrics.Precision(), tf.keras.metrics.Recall()])
    
    # logName = generate_logs_Name("base_model", logFormat=".log", clear_folder=True)
    # csvLogger = callbacks.CSVLogger(filename=logName)
    tensorBoard = callbacks.TensorBoard(log_dir="logs", histogram_freq=1)

    history = model.fit(train_data,
                        validation_data=val_data,
                        epochs=epochs,
                        callbacks=[tensorBoard],
                        verbose=1)
    return history

In [22]:
baseModel = create_base_model()
baseModel.summary()

Model: "cnn_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 131072)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │    16,777,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,779,530 (64.01 MB)

 Trainable params: 16,779,530 (64.01 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history = start_model_training(model=baseModel,
                               train_data=Train_scaled,
                               val_data=Val_scaled,
                               epochs=15)

Epoch 1/15


InvalidArgumentError: Graph execution error:

Detected at node LogicalAnd defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "c:\Study\gitRepo\CV_Amazon_Cloth_Classify\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "c:\Study\gitRepo\CV_Amazon_Cloth_Classify\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "c:\Study\gitRepo\CV_Amazon_Cloth_Classify\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start

  File "c:\Study\gitRepo\CV_Amazon_Cloth_Classify\.venv\Lib\site-packages\tornado\platform\asyncio.py", line 205, in start

  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2544.0_x64__qbz5n2kfra8p0\Lib\asyncio\base_events.py", line 645, in run_forever

  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2544.0_x64__qbz5n2kfra8p0\Lib\asyncio\base